# FAERS 2026Q2 -- Exploratory Data Analysis (sample)

Scope: the **5,000-case sample** parsed from `1_ADR26Q2.xml` and staged in
`data/sample/faers_sample.duckdb` (see `ingestion/run_ingest.py` /
`ingestion/staging_db.py`). Re-run this notebook end-to-end after scaling up
(`data/staging/faers_staging.duckdb`, all 422,459 cases) to refresh every
number and chart against the full dataset -- see the last cell for the toggle.

**Standing caveat** (same one the chatbot appends to every answer): this is
spontaneous adverse-event report data. Counts here describe *what was
reported*, not incidence, risk, or causality, and are not adjusted for
exposure or reporting bias.

## Setup

In [ ]:
import sys, os, json
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

# Toggle this to run the same notebook against the full-scale dataset once
# scripts\run_full_pipeline.ps1 has been run.
MODE = "sample"  # "sample" or "full"

from config.settings import settings  # noqa: E402
from ontology.code_lookups import (  # noqa: E402
    DRUG_CHARACTERIZATION, PATIENT_AGE_GROUP, PATIENT_SEX, QUALIFICATION,
    REACTION_OUTCOME, REPORT_TYPE, ACTION_DRUG, DECHALLENGE,
)

db_path = (settings.sample_dir if MODE == "sample" else settings.staging_dir) / (
    "faers_sample.duckdb" if MODE == "sample" else "faers_staging.duckdb"
)
con = duckdb.connect(str(db_path), read_only=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

def q(sql, **params):
    """Run SQL against the staging DB, return a DataFrame."""
    return con.execute(sql, params).fetchdf() if params else con.sql(sql).df()

def barh(series, title, xlabel, n=20, figsize=(8, 6)):
    top = series.head(n).iloc[::-1]
    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(top.index.astype(str), top.values, color="#1e6e63")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    fig.tight_layout()
    return fig, ax

print(f"connected to {db_path}  (mode={MODE})")

## 1. Dataset overview

In [ ]:
manifest_path = db_path.parent / "run_manifest.json"
if manifest_path.exists():
    print(json.dumps(json.loads(manifest_path.read_text()), indent=2))

In [ ]:
for table in ["cases", "case_drugs", "case_reactions"]:
    n = q(f"SELECT COUNT(*) AS n FROM {table}").iloc[0, 0]
    print(f"{table:<16} {n:>10,} rows")

In [ ]:
q("SELECT * FROM cases LIMIT 3")

In [ ]:
q("SELECT * FROM case_drugs LIMIT 3")

In [ ]:
q("SELECT * FROM case_reactions LIMIT 3")

## 2. Data completeness

FDA's own FAQ warns that most fields are voluntary and frequently blank.
Worth seeing exactly how sparse before drawing any conclusion from a field.

In [ ]:
case_fields = [
    "primarysourcecountry", "occurcountry", "receivedate", "reporttype",
    "serious", "fulfillexpeditecriteria", "qualification", "senderorganization",
    "patientonsetage_years", "patientagegroup", "patientweight", "patientsex",
    "event_date_from_narrative",
]
total = q("SELECT COUNT(*) AS n FROM cases").iloc[0, 0]
rows = []
for f in case_fields:
    non_null = q(f"SELECT COUNT({f}) AS n FROM cases").iloc[0, 0]
    rows.append({"field": f, "pct_filled": round(100 * non_null / total, 1)})
completeness = pd.DataFrame(rows).sort_values("pct_filled")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(completeness["field"], completeness["pct_filled"], color="#1e6e63")
ax.set_xlim(0, 100)
ax.set_xlabel("% of cases with a value")
ax.set_title("Case-field completeness")
fig.tight_layout()
plt.show()
completeness

## 3. Patient demographics

In [ ]:
ages = q("SELECT patientonsetage_years FROM cases WHERE patientonsetage_years IS NOT NULL")
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ages["patientonsetage_years"], bins=40, color="#1e6e63", edgecolor="white")
ax.set_title(f"Patient age at onset (n={len(ages):,} reports with an age recorded)")
ax.set_xlabel("years")
ax.set_ylabel("reports")
fig.tight_layout()
plt.show()
ages["patientonsetage_years"].describe()

In [ ]:
sex_counts = q("SELECT patientsex, COUNT(*) AS n FROM cases GROUP BY patientsex ORDER BY n DESC")
sex_counts["label"] = sex_counts["patientsex"].map(PATIENT_SEX).fillna("Not recorded")
sex_counts.set_index("label")["n"].plot.bar(figsize=(5, 4), color="#1e6e63", rot=0, title="Reports by patient sex")
plt.ylabel("reports")
plt.tight_layout()
plt.show()
sex_counts[["label", "n"]]

In [ ]:
age_grp = q("SELECT patientagegroup, COUNT(*) AS n FROM cases GROUP BY patientagegroup ORDER BY n DESC")
age_grp["label"] = age_grp["patientagegroup"].map(PATIENT_AGE_GROUP).fillna("Not recorded")
age_grp.set_index("label")["n"].plot.bar(figsize=(6, 4), color="#1e6e63", rot=30, title="Reports by patient age group")
plt.ylabel("reports")
plt.tight_layout()
plt.show()

In [ ]:
country = q(
    "SELECT primarysourcecountry, COUNT(*) AS n FROM cases "
    "WHERE primarysourcecountry IS NOT NULL GROUP BY primarysourcecountry ORDER BY n DESC LIMIT 15"
).set_index("primarysourcecountry")["n"]
barh(country, "Top 15 primary source countries", "reports", n=15, figsize=(7, 5))
plt.show()

In [ ]:
qual = q("SELECT qualification, COUNT(*) AS n FROM cases GROUP BY qualification ORDER BY n DESC")
qual["label"] = qual["qualification"].map(QUALIFICATION).fillna("Not recorded")
qual.set_index("label")["n"].plot.barh(figsize=(7, 4), color="#1e6e63", title="Reporter qualification")
plt.xlabel("reports")
plt.tight_layout()
plt.show()

In [ ]:
rt = q("SELECT reporttype, COUNT(*) AS n FROM cases GROUP BY reporttype ORDER BY n DESC")
rt["label"] = rt["reporttype"].map(REPORT_TYPE).fillna("Not recorded")
rt[["label", "n"]]

In [ ]:
by_date = q(
    "SELECT substr(receivedate, 1, 6) AS year_month, COUNT(*) AS n FROM cases "
    "WHERE receivedate IS NOT NULL AND length(receivedate) >= 6 "
    "GROUP BY year_month ORDER BY year_month"
)
by_date.set_index("year_month")["n"].plot.bar(figsize=(7, 4), color="#1e6e63", rot=45, title="Reports by FDA receive month")
plt.ylabel("reports")
plt.tight_layout()
plt.show()

## 4. Seriousness

In [ ]:
serious_flags = {
    "serious": "Overall serious",
    "seriousnessdeath": "Death",
    "seriousnesshospitalization": "Hospitalization",
    "seriousnesslifethreatening": "Life-threatening",
    "seriousnessdisabling": "Disabling",
    "seriousnesscongenitalanomali": "Congenital anomaly",
    "seriousnessother": "Other serious",
}
rows = []
for col, label in serious_flags.items():
    n_yes = q(f"SELECT COUNT(*) AS n FROM cases WHERE {col} = '1'").iloc[0, 0]
    rows.append({"flag": label, "reports": n_yes, "pct_of_total": round(100 * n_yes / total, 1)})
seriousness = pd.DataFrame(rows).sort_values("reports", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(seriousness["flag"][::-1], seriousness["reports"][::-1], color="#a9672c")
ax.set_xlabel("reports")
ax.set_title(f"Seriousness flags (of {total:,} total reports)")
fig.tight_layout()
plt.show()
seriousness

In [ ]:
by_age = q(
    "SELECT patientagegroup, "
    "  COUNT(*) AS n, "
    "  SUM(CASE WHEN serious = '1' THEN 1 ELSE 0 END) AS serious_n "
    "FROM cases WHERE patientagegroup IS NOT NULL GROUP BY patientagegroup"
)
by_age["label"] = by_age["patientagegroup"].map(PATIENT_AGE_GROUP)
by_age["pct_serious"] = round(100 * by_age["serious_n"] / by_age["n"], 1)
by_age.sort_values("pct_serious", ascending=False)[["label", "n", "serious_n", "pct_serious"]]

## 5. Drugs

In [ ]:
top_drugs = q(
    """
    WITH exploded AS (
        SELECT case_id, upper(trim(x)) AS drug_key
        FROM case_drugs, UNNEST(active_ingredients) AS t(x)
        WHERE trim(x) != ''
        UNION ALL
        SELECT case_id, upper(trim(medicinalproduct)) AS drug_key
        FROM case_drugs
        WHERE len(active_ingredients) = 0 AND medicinalproduct IS NOT NULL AND trim(medicinalproduct) != ''
    )
    SELECT drug_key, COUNT(DISTINCT case_id) AS reports
    FROM exploded GROUP BY drug_key ORDER BY reports DESC LIMIT 20
    """
).set_index("drug_key")["reports"]
barh(top_drugs, "Top 20 drugs by distinct report count", "reports", n=20, figsize=(8, 7))
plt.show()

In [ ]:
role = q("SELECT drugcharacterization, COUNT(*) AS n FROM case_drugs GROUP BY drugcharacterization ORDER BY n DESC")
role["label"] = role["drugcharacterization"].map(DRUG_CHARACTERIZATION).fillna("Not recorded")
role.set_index("label")["n"].plot.bar(figsize=(6, 4), color="#1e6e63", rot=20, title="Drug role in report (suspect / concomitant / ...)")
plt.ylabel("drug rows")
plt.tight_layout()
plt.show()

In [ ]:
routes = q(
    "SELECT drugadministrationroute, COUNT(*) AS n FROM case_drugs "
    "WHERE drugadministrationroute IS NOT NULL GROUP BY drugadministrationroute ORDER BY n DESC LIMIT 10"
).set_index("drugadministrationroute")["n"]
barh(routes, "Top 10 administration route codes", "drug rows", n=10, figsize=(6, 4))
plt.show()

In [ ]:
drugs_per_case = q("SELECT case_id, COUNT(*) AS n_drugs FROM case_drugs GROUP BY case_id")
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(drugs_per_case["n_drugs"], bins=range(1, drugs_per_case["n_drugs"].max() + 2), color="#1e6e63", edgecolor="white", align="left")
ax.set_title("Drugs listed per case")
ax.set_xlabel("drug rows in the case")
ax.set_ylabel("cases")
fig.tight_layout()
plt.show()
drugs_per_case["n_drugs"].describe()

In [ ]:
# how often did we have to fall back from active_ingredients to medicinalproduct?
n_drug_rows = q("SELECT COUNT(*) AS n FROM case_drugs").iloc[0, 0]
n_with_substance = q("SELECT COUNT(*) AS n FROM case_drugs WHERE len(active_ingredients) > 0").iloc[0, 0]
n_fallback = q(
    "SELECT COUNT(*) AS n FROM case_drugs "
    "WHERE len(active_ingredients) = 0 AND medicinalproduct IS NOT NULL AND trim(medicinalproduct) != ''"
).iloc[0, 0]
n_neither = n_drug_rows - n_with_substance - n_fallback
print(f"active_ingredients present : {n_with_substance:>6,} rows ({100*n_with_substance/n_drug_rows:.1f}%)")
print(f"medicinalproduct fallback  : {n_fallback:>6,} rows ({100*n_fallback/n_drug_rows:.1f}%)")
print(f"neither (dropped from graph, per FAQ #24 guidance): {n_neither:>6,} rows ({100*n_neither/n_drug_rows:.1f}%)")

## 6. Reactions

In [ ]:
top_reactions = q(
    "SELECT reactionmeddrapt, COUNT(*) AS n FROM case_reactions "
    "WHERE reactionmeddrapt IS NOT NULL GROUP BY reactionmeddrapt ORDER BY n DESC LIMIT 20"
).set_index("reactionmeddrapt")["n"]
barh(top_reactions, "Top 20 most-reported reactions", "reports", n=20, figsize=(8, 7))
plt.show()

In [ ]:
outcome = q("SELECT reactionoutcome, COUNT(*) AS n FROM case_reactions GROUP BY reactionoutcome ORDER BY n DESC")
outcome["label"] = outcome["reactionoutcome"].map(REACTION_OUTCOME).fillna("Not recorded")
outcome.set_index("label")["n"].plot.barh(figsize=(7, 4), color="#a9672c", title="Reaction outcome at last observation")
plt.xlabel("reaction rows")
plt.tight_layout()
plt.show()

In [ ]:
reactions_per_case = q("SELECT case_id, COUNT(*) AS n_reactions FROM case_reactions GROUP BY case_id")
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(reactions_per_case["n_reactions"], bins=range(1, reactions_per_case["n_reactions"].max() + 2), color="#a9672c", edgecolor="white", align="left")
ax.set_title("Reactions listed per case")
ax.set_xlabel("reaction rows in the case")
ax.set_ylabel("cases")
fig.tight_layout()
plt.show()
reactions_per_case["n_reactions"].describe()

In [ ]:
q("SELECT reactionmeddraversionpt, COUNT(*) AS n FROM case_reactions GROUP BY reactionmeddraversionpt ORDER BY n DESC")

## 7. Drug x reaction co-occurrence

Same relationship the graph's `REPORTED_WITH` edges encode (`graph_load/postload_aggregates.cypher`)
-- computed here directly from the staging tables as an independent check. If
you've loaded the graph, spot-check a row or two against Neo4j Browser:
`MATCH (d:Drug {drug_key:'...'})-[r:REPORTED_WITH]->(react:Reaction) RETURN react.pt, r.case_count ORDER BY r.case_count DESC LIMIT 10`.

In [ ]:
pairs = q(
    """
    WITH drug_exploded AS (
        SELECT DISTINCT case_id, upper(trim(x)) AS drug_key
        FROM case_drugs, UNNEST(active_ingredients) AS t(x) WHERE trim(x) != ''
        UNION
        SELECT DISTINCT case_id, upper(trim(medicinalproduct)) AS drug_key
        FROM case_drugs
        WHERE len(active_ingredients) = 0 AND medicinalproduct IS NOT NULL AND trim(medicinalproduct) != ''
    )
    SELECT de.drug_key, cr.reactionmeddrapt AS reaction, COUNT(DISTINCT de.case_id) AS case_count
    FROM drug_exploded de
    JOIN case_reactions cr ON cr.case_id = de.case_id
    WHERE cr.reactionmeddrapt IS NOT NULL
    GROUP BY de.drug_key, cr.reactionmeddrapt
    ORDER BY case_count DESC LIMIT 15
    """
)
pairs

## 8. Data-quality checks

In [ ]:
delete_path = REPO_ROOT / "faers_xml_2026q2" / "Deleted" / "DELETE26Q2.txt"
delete_ids = {
    line.strip() for line in delete_path.read_text().splitlines() if line.strip().isdigit()
}
loaded_ids = set(q("SELECT safetyreportid FROM cases")["safetyreportid"])
overlap = delete_ids & loaded_ids
print(f"delete-list IDs           : {len(delete_ids):,}")
print(f"IDs present in this sample: {len(loaded_ids):,}")
print(f"overlap (should be ~0, per PLAN.md \u00a72.2): {len(overlap)}")

In [ ]:
dup_check = q("SELECT safetyreportid, COUNT(*) AS n FROM cases GROUP BY safetyreportid HAVING COUNT(*) > 1")
print(f"duplicate case IDs in staging: {len(dup_check)}")

In [ ]:
date_fmt = q(
    "SELECT drugstartdateformat, COUNT(*) AS n FROM case_drugs "
    "WHERE drugstartdateformat IS NOT NULL GROUP BY drugstartdateformat ORDER BY n DESC"
)
fmt_labels = {"102": "day (YYYYMMDD)", "610": "month (YYYYMM)", "602": "year (YYYY)"}
date_fmt["precision"] = date_fmt["drugstartdateformat"].map(fmt_labels).fillna(date_fmt["drugstartdateformat"])
n_with_date = q("SELECT COUNT(*) AS n FROM case_drugs WHERE drugstartdate IS NOT NULL").iloc[0, 0]
n_total_drugs = q("SELECT COUNT(*) AS n FROM case_drugs").iloc[0, 0]
print(f"drug rows with a start date: {n_with_date:,} / {n_total_drugs:,} ({100*n_with_date/n_total_drugs:.1f}%)")
date_fmt[["precision", "n"]]

In [ ]:
age_unit = q(
    "SELECT patientonsetageunit, COUNT(*) AS n FROM cases "
    "WHERE patientonsetageunit IS NOT NULL GROUP BY patientonsetageunit ORDER BY n DESC"
)
unit_labels = {"800": "Decade", "801": "Year", "802": "Month", "803": "Week", "804": "Day", "805": "Hour"}
age_unit["unit"] = age_unit["patientonsetageunit"].map(unit_labels)
age_unit[["unit", "n"]]

In [ ]:
missing_drug_name = q(
    "SELECT COUNT(*) AS n FROM case_drugs "
    "WHERE (medicinalproduct IS NULL OR trim(medicinalproduct) = '') AND len(active_ingredients) = 0"
).iloc[0, 0]
print(f"drug rows with NO name at all (medicinalproduct blank AND no active ingredient): {missing_drug_name}")
print("Per FAQ #24, FDA says to disregard these -- our graph loader already excludes them from INVOLVES_DRUG.")

## 9. Summary

In [ ]:
top_drug_row = top_drugs.head(1)
top_reaction_row = top_reactions.head(1)
top_pair = pairs.iloc[0]
pct_serious = seriousness.loc[seriousness["flag"] == "Overall serious", "pct_of_total"].iloc[0]
pct_death = seriousness.loc[seriousness["flag"] == "Death", "pct_of_total"].iloc[0]

print(f"""Key numbers for this {MODE} run ({total:,} cases):

- Most-reported drug     : {top_drug_row.index[0]}  ({int(top_drug_row.iloc[0]):,} reports)
- Most-reported reaction : {top_reaction_row.index[0]}  ({int(top_reaction_row.iloc[0]):,} reports)
- Top co-reported pair   : {top_pair['drug_key']} + {top_pair['reaction']}  ({int(top_pair['case_count'])} cases)
- Marked overall serious : {pct_serious}% of reports
- Resulted in death      : {pct_death}% of reports
- Median drugs per case  : {drugs_per_case['n_drugs'].median():.0f}
- Median reactions/case  : {reactions_per_case['n_reactions'].median():.0f}
- Drug-name fallback rate: {100*n_fallback/n_drug_rows:.1f}% of drug rows had no active substance and used the verbatim product name instead

Reminder: these are counts of spontaneous reports, not incidence or risk --
see PLAN.md \u00a77 for what this data can and can't support.""")